# FIFA World Cup 2026 — Linear Regression 2.1
## 03. Regression Modelling

**Objective:** Predict home-team `goal_difference` from the same eight historical predictors used in Data Preparation and EDA.

This notebook follows the modelling stages of the supplied Regression 2.2 reference: data loading, splitting, baseline modelling, diagnostics, regularised regression, comparison and final evaluation. It does **not** fill or reconstruct the first 24 rows. Run your **original** Data Preparation notebook first to create `Regression_2_1_Prepared_Data.csv` (not the historical backfill version).

## Step 1 — Import libraries and load the prepared dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import TimeSeriesSplit, cross_validate, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import os



In [ ]:
pd.set_option("display.max_columns", None)
file_path = Path("Regression_2_1_Prepared_Data.csv")
if not file_path.exists():
    raise FileNotFoundError("Run the ORIGINAL Regression 2.1 Data Preparation notebook first.")
df = pd.read_csv(file_path)
df["date"] = pd.to_datetime(df["date"], errors="raise")
df = df.sort_values(["date", "match_id"], kind="stable").reset_index(drop=True)
print("Dataset shape:", df.shape)
display(df.head())

NameError: name 'pd' is not defined

: 

## Step 2 — Define the target and eight predictors

In [ ]:
target = "goal_difference"
features = [
    "Team_prior_matches", "Opponent_prior_matches",
    "Team_prior_goals_scored_avg", "Opponent_prior_goals_scored_avg",
    "Team_rest_days", "Opponent_rest_days",
    "Team_prior_win_rate", "Opponent_prior_win_rate"
]
assert len(features) == 8 and len(set(features)) == 8
assert set(features + [target, "match_id", "date"]).issubset(df.columns)
assert df["match_id"].is_unique, "Expected one observation per match."
X = df[features].apply(pd.to_numeric, errors="raise")
y = pd.to_numeric(df[target], errors="raise")
assert y.notna().all()
print("Target:", target)
print("Predictors:", features)
print("Missing values:")
display(X.isna().sum().to_frame("Missing"))

### Note on the original first-round values

The original dataset may use zeros as placeholders where a team has no prior tournament matches. These values are **retained exactly as supplied**, not filled from `results.csv`. This limitation should be disclosed in the report. A zero placeholder may be substantively different from a genuine zero-day rest period.

## Step 3 — Chronological train–test split

In [ ]:
# Earlier fixtures are used for training; the final 20% are held out.
cutoff = int(len(df) * 0.80)
X_train, X_test = X.iloc[:cutoff], X.iloc[cutoff:]
y_train, y_test = y.iloc[:cutoff], y.iloc[cutoff:]
train_info, test_info = df.iloc[:cutoff], df.iloc[cutoff:]
assert train_info["date"].max() <= test_info["date"].min()
assert set(train_info["match_id"]).isdisjoint(test_info["match_id"])
print("Training observations:", len(X_train), "| Test observations:", len(X_test))
print("Train dates:", train_info["date"].min(), "to", train_info["date"].max())
print("Test dates:", test_info["date"].min(), "to", test_info["date"].max())

The test set is held out until model selection is complete. A chronological split is used because the objective is to predict later matches from earlier information. If several matches share the exact same date and no reliable kickoff order is available, check the boundary manually.

## Step 4 — Baseline Multiple Linear Regression

In [ ]:
def make_pipeline(regressor):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", regressor)
    ])

baseline_model = make_pipeline(LinearRegression())
# Time-series CV respects observation order. Scaling and imputation are fitted inside each fold.
cv = TimeSeriesSplit(n_splits=5)
scoring = {"RMSE": "neg_root_mean_squared_error", "MAE": "neg_mean_absolute_error", "R2": "r2"}
base_cv = cross_validate(baseline_model, X_train, y_train, cv=cv, scoring=scoring)
baseline_model.fit(X_train, y_train)
print("Baseline training RMSE:", round(np.sqrt(mean_squared_error(y_train, baseline_model.predict(X_train))), 3))
print("Baseline CV RMSE:", round(-base_cv["test_RMSE"].mean(), 3))
print("Baseline CV MAE:", round(-base_cv["test_MAE"].mean(), 3))
print("Baseline CV R²:", round(base_cv["test_R2"].mean(), 3))

## Step 5 — Baseline diagnostics

In [ ]:
# Out-of-fold predictions: only predict validation folds; initial training block has no CV prediction.
oof_pred = pd.Series(np.nan, index=X_train.index, dtype=float)
for tr_idx, val_idx in cv.split(X_train):
    fold_model = make_pipeline(LinearRegression())
    fold_model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    oof_pred.iloc[val_idx] = fold_model.predict(X_train.iloc[val_idx])
valid = oof_pred.notna()
observed = y_train.loc[valid]
predicted = oof_pred.loc[valid]
residuals = observed - predicted
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0,0].scatter(observed, predicted, alpha=.7)
lo = min(observed.min(), predicted.min()); hi = max(observed.max(), predicted.max())
axes[0,0].plot([lo, hi], [lo, hi], '--', color='black')
axes[0,0].set(xlabel='Actual goal difference', ylabel='Predicted goal difference', title='Out-of-fold predictions')
axes[0,1].scatter(predicted, residuals, alpha=.7)
axes[0,1].axhline(0, linestyle='--', color='black')
axes[0,1].set(xlabel='Predicted goal difference', ylabel='Residual', title='Residuals vs fitted')
axes[1,0].hist(residuals, bins=12)
axes[1,0].set(xlabel='Residual', ylabel='Frequency', title='Residual distribution')
stats.probplot(residuals, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Residual Q-Q plot')
plt.tight_layout(); plt.show()
print('Out-of-fold RMSE:', round(np.sqrt(mean_squared_error(observed, predicted)), 3))

In [ ]:
# Check multicollinearity on training data only.
X_imp = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X_train), columns=features)
X_vif = sm.add_constant(X_imp, has_constant='add')
vif_table = pd.DataFrame({
    'Predictor': features,
    'VIF': [variance_inflation_factor(X_vif.values, i + 1) for i in range(len(features))]
}).sort_values('VIF', ascending=False)
display(vif_table)

coefficients = pd.DataFrame({
    'Predictor': features,
    'Standardised coefficient': baseline_model.named_steps['regressor'].coef_
}).sort_values('Standardised coefficient', key=lambda x: x.abs(), ascending=False)
display(coefficients)

### Diagnostic interpretation

Inspect whether residuals show curvature, changing spread or extreme values. VIF indicates overlapping explanatory information; it does not automatically justify deleting a predictor. Unlike goal counts, **goal difference can be negative**, so negative predictions are not inherently invalid.

## Step 6 — Tune Ridge, Lasso and Elastic Net

In [ ]:
alpha_grid = np.logspace(-3, 2, 25)
searches = {
    'Ridge': GridSearchCV(make_pipeline(Ridge()), {'regressor__alpha': alpha_grid}, cv=cv, scoring=scoring, refit='RMSE'),
    'Lasso': GridSearchCV(make_pipeline(Lasso(max_iter=50000)), {'regressor__alpha': alpha_grid}, cv=cv, scoring=scoring, refit='RMSE'),
    'Elastic Net': GridSearchCV(make_pipeline(ElasticNet(max_iter=50000)), {
        'regressor__alpha': alpha_grid, 'regressor__l1_ratio': [0.2, 0.5, 0.8]
    }, cv=cv, scoring=scoring, refit='RMSE')
}
for name, search in searches.items():
    search.fit(X_train, y_train)
    print(name, 'best parameters:', search.best_params_)

## Step 7 — Compare models using training-only cross-validation

In [ ]:
comparison_rows = [{
    'Model': 'Linear Regression',
    'CV_RMSE': -base_cv['test_RMSE'].mean(),
    'CV_MAE': -base_cv['test_MAE'].mean(),
    'CV_R2': base_cv['test_R2'].mean()
}]
for name, search in searches.items():
    i = search.best_index_
    comparison_rows.append({
        'Model': name,
        'CV_RMSE': -search.cv_results_['mean_test_RMSE'][i],
        'CV_MAE': -search.cv_results_['mean_test_MAE'][i],
        'CV_R2': search.cv_results_['mean_test_R2'][i]
    })
cv_comparison = pd.DataFrame(comparison_rows).sort_values('CV_RMSE').reset_index(drop=True)
display(cv_comparison.round(3))
cv_comparison.plot.bar(x='Model', y='CV_RMSE', legend=False, figsize=(8, 4))
plt.ylabel('Cross-validation RMSE (goal difference)')
plt.title('Model comparison — lower is better')
plt.tight_layout(); plt.show()

## Step 8 — Select the model and evaluate the untouched test set

In [ ]:
best_name = cv_comparison.loc[0, 'Model']
models = {'Linear Regression': baseline_model, **{name: search.best_estimator_ for name, search in searches.items()}}
final_model = models[best_name]
final_predictions = final_model.predict(X_test)
mean_baseline_prediction = np.repeat(y_train.mean(), len(y_test))

def metrics(actual, pred):
    return {'MAE': mean_absolute_error(actual, pred),
            'MSE': mean_squared_error(actual, pred),
            'RMSE': np.sqrt(mean_squared_error(actual, pred)),
            'R2': r2_score(actual, pred)}

final_results = pd.DataFrame([
    {'Model': best_name, **metrics(y_test, final_predictions)},
    {'Model': 'Training-mean baseline', **metrics(y_test, mean_baseline_prediction)}
])
print('Selected model:', best_name)
display(final_results.round(4))

## Step 9 — Final predictions, residuals and coefficients

In [ ]:
prediction_table = test_info[['match_id', 'date', 'Home_Team', 'Away_Team']].copy()
prediction_table['Actual_Goal_Difference'] = y_test.to_numpy()
prediction_table['Predicted_Goal_Difference'] = final_predictions
prediction_table['Residual'] = y_test.to_numpy() - final_predictions
display(prediction_table.head(10))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(y_test, final_predictions, alpha=.7)
lo = min(y_test.min(), final_predictions.min()); hi = max(y_test.max(), final_predictions.max())
axes[0].plot([lo, hi], [lo, hi], '--', color='black')
axes[0].set(xlabel='Actual goal difference', ylabel='Predicted goal difference', title='Final test predictions')
axes[1].scatter(final_predictions, prediction_table['Residual'], alpha=.7)
axes[1].axhline(0, linestyle='--', color='black')
axes[1].set(xlabel='Predicted goal difference', ylabel='Residual', title='Test residuals')
plt.tight_layout(); plt.show()
final_coefficients = pd.DataFrame({
    'Predictor': features,
    'Standardised coefficient': final_model.named_steps['regressor'].coef_
}).sort_values('Standardised coefficient', key=lambda x: x.abs(), ascending=False)
display(final_coefficients)

## Step 10 — Save results and final validation

In [ ]:
cv_comparison.to_csv('Regression_2_1_Model_Comparison.csv', index=False)
prediction_table.to_csv('Regression_2_1_Test_Predictions.csv', index=False)
final_coefficients.to_csv('Regression_2_1_Model_Coefficients.csv', index=False)
assert len(features) == 8
assert len(X_train) + len(X_test) == len(df)
assert np.isfinite(final_predictions).all()
assert set(train_info['match_id']).isdisjoint(test_info['match_id'])
print('Validation passed. Saved comparison, test predictions and coefficients.')

## Modelling conclusion — complete after running

Report the selected model, its cross-validation RMSE and final test MAE/RMSE/R². Compare it with ordinary Multiple Linear Regression and the training-mean baseline. Discuss the eight predictor coefficients and any residual patterns. Explain that the initial historical placeholders have not yet been corrected and that a small single-tournament sample limits generalisation.

**Do not copy performance figures from Regression 2.2:** this notebook uses a different target (`goal_difference`), eight different predictor fields and a chronological evaluation.